# Aplikasi Rekomendasi Citra Serupa — Tahap 2: metode klasik

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t5/blob/main/notebooks/02_metode_klasik.ipynb)

Kelompok ini wajib **2 metode klasik**. Pilihan yang disarankan:

- Histogram warna (HSV)
- Tekstur GLCM
- HOG

Setiap metode adalah satu fungsi. Evaluasi memakai `pcd_proyek.evaluasi`, sehingga metrik sama dengan kelompok lain.

In [24]:
# Siapkan lingkungan: salin repo ini dan hubungkan Google Drive kelompok.
import pathlib
import subprocess
import sys

import cv2
import numpy as np

REPO = "Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t5"
try:
    import google.colab  # noqa: F401
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    AKAR = pathlib.Path("/content/repo")
    if AKAR.exists():
        subprocess.run(["git", "-C", str(AKAR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(AKAR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER_KERJA = pathlib.Path("/content/drive/MyDrive/pcd2026-ti23ka-t5")
    if not FOLDER_KERJA.exists():
        raise FileNotFoundError(f"{FOLDER_KERJA} tidak ada. Tambahkan pintasan folder Drive kelompok ke Drive Saya (lihat PANDUAN.md).")
else:  # dijalankan di komputer sendiri dari folder notebooks/
    AKAR = pathlib.Path.cwd().resolve().parent
    FOLDER_KERJA = AKAR / "data"
sys.path.insert(0, str(AKAR))

from pcd_proyek import evaluasi as ev, metrik, util  # noqa: E402

FOLDER_CITRA = FOLDER_KERJA / "citra" if (FOLDER_KERJA / "citra").exists() else AKAR / "data" / "sample"
BERKAS_ANOTASI = FOLDER_KERJA / "anotasi.csv"
FOLDER_FIGUR = FOLDER_KERJA / "figures"
BERKAS_HASIL = FOLDER_KERJA / "hasil_metode.json"
KOLOM_ANOTASI = ['berkas', 'kategori']
JUMLAH_ANGGOTA = 4
print("Citra   :", FOLDER_CITRA)
print("Anotasi :", BERKAS_ANOTASI, "(ada)" if BERKAS_ANOTASI.exists() else "(belum ada)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Citra   : /content/drive/MyDrive/pcd2026-ti23ka-t5/citra
Anotasi : /content/drive/MyDrive/pcd2026-ti23ka-t5/anotasi.csv (ada)


In [25]:
anotasi = util.baca_anotasi(BERKAS_ANOTASI, KOLOM_ANOTASI)
UJI_CEPAT = None  # isi angka (mis. 20) saat mencoba-coba agar cepat; kembalikan ke None untuk hasil akhir
print(len(anotasi), "baris anotasi")

200 baris anotasi


Histogram Warna HSV 3D. Minuman (kopi, teh, mocktails, milkshake) memiliki ciri visual utama berupa distribusi warna yang khas.

In [26]:
def metode_klasik_1(citra_bgr):
    """Metode klasik 1: Histogram Warna HSV 3D."""
    # 1. Konversi ke HSV
    hsv = cv2.cvtColor(citra_bgr, cv2.COLOR_BGR2HSV)

    # 2. Hitung histogram 3D (8x8x8 bin = 512 dimensi)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256])

    # 3. Normalisasi
    cv2.normalize(hist, hist)

    return hist.flatten().astype(np.float64)

In [27]:
hasil_1 = ev.evaluasi_temu_kembali(metode_klasik_1, anotasi, FOLDER_CITRA, k=5, batas=UJI_CEPAT)
print(hasil_1['metrik'])
NAMA_1 = "Histogram HSV (8x8x8)"
if UJI_CEPAT is None:
    ev.simpan_hasil(BERKAS_HASIL, NAMA_1, "klasik", hasil_1)

{'Presisi@5': 0.295, 'mAP': 0.2827896612060389, 'jumlah': 200}


## Metode klasik 2

**Nama metode dan alasannya:** Tekstur GLCM (Gray-Level Co-occurrence Matrix). Karakteristik fisik minuman seperti kejernihan teh, kepekatan kopi, busa/foam milkshake, dan lapisan es pada mocktail memiliki ciri tekstur visual yang khas.


In [28]:
from skimage.feature import graycomatrix, graycoprops

def metode_klasik_2(citra_bgr):
    """Metode klasik 2: Tekstur GLCM.

    Masukan: citra BGR uint8. Keluaran: vektor_fitur — vektor 1D numpy.
    """
    # 1. Ubah citra ke grayscale
    gray = cv2.cvtColor(citra_bgr, cv2.COLOR_BGR2GRAY)

    # 2. Perkecil ke 128x128 agar komputasi cepat
    gray_kecil = cv2.resize(gray, (128, 128), interpolation=cv2.INTER_AREA)

    # 3. Kuantisasi ke 16 level abu-abu
    gray_16 = (gray_kecil // 16).astype(np.uint8)

    # 4. Hitung GLCM pada jarak 1 & 3 piksel dengan 4 sudut
    glcm = graycomatrix(gray_16, distances=[1, 3], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=16, symmetric=True, normed=True)

    # 5. Ekstrak statistik fitur tekstur
    fitur = []
    for prop in ('contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM'):
        fitur.extend(graycoprops(glcm, prop).flatten())

    return np.array(fitur, dtype=np.float64)

In [29]:
hasil_2 = ev.evaluasi_temu_kembali(metode_klasik_2, anotasi, FOLDER_CITRA, k=5, batas=UJI_CEPAT)
print(hasil_2['metrik'])
NAMA_2 = "Tekstur GLCM"
if UJI_CEPAT is None:
    ev.simpan_hasil(BERKAS_HASIL, NAMA_2, "klasik", hasil_2)

{'Presisi@5': 0.274, 'mAP': 0.28385930633971246, 'jumlah': 200}


## Simpan ke GitHub

1. **Runtime → Restart session and run all**, pastikan tidak ada error.
2. **File → Download → Download .ipynb**, lalu di github.com pilih **branch Anda** → folder `notebooks/` → **Add file → Upload files** → commit ke branch Anda. (Bila menu Colab menampilkan *Save a copy in GitHub*, itu juga boleh. Jangan pakai *GitHub Gist*.)
3. Buka *pull request* ke `main` di github.com. Langkah lengkap: `PANDUAN.md`.